# Camada Silver — `ecommerce_clientes` 

Este notebook lê os micro-lotes da Bronze no SQL Server, aplica as 5 regras de qualidade em colunas booleanas e grava a Silver em `append`.

Controle de idempotência:
- A Bronze possui `bronze_source_file`.
- A Silver mantém `bronze_source_file`.
- Antes de processar, o notebook consulta quais arquivos já existem na Silver.
- Apenas arquivos ainda não processados são enviados para `squad1.silver_ecommerce_cliente`.

##  Imports e parâmetros

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce
import uuid

RUN_ID = str(uuid.uuid4())
TABELA_SILVER = "squad1.silver_ecommerce_clientes"
TABELA_DQ_LOGS = "squad1.dq_monitoring_logs"
TABELA_BRONZE_CLIENTES = "squad1.bronze_ecommerce_clientes"
TABELA_BRONZE_ENDERECOS = "squad1.bronze_ecommerce_enderecos"
TABELA_BRONZE_PEDIDOS = "squad1.bronze_ecommerce_pedidos"

print("RUN_ID:", RUN_ID)


## Funções auxiliares



In [0]:
def tabela_existe(nome_tabela: str) -> bool:
    try:
        return spark.catalog.tableExists(nome_tabela)
    except Exception:
        try:
            spark.table(nome_tabela).limit(1).count()
            return True
        except Exception:
            return False


def ler_tabela_delta(nome_tabela: str):
    if not tabela_existe(nome_tabela):
        raise Exception(f"Tabela Delta não encontrada: {nome_tabela}")
    return spark.table(nome_tabela)


def anti_duplicidade_por_arquivo(df_novo, tabela_destino: str, coluna_arquivo: str = "bronze_source_file"):
    if not tabela_existe(tabela_destino):
        print(f"Tabela {tabela_destino} ainda não existe. Todo micro-lote será processado.")
        return df_novo

    df_processados = (
        spark.table(tabela_destino)
        .select(coluna_arquivo)
        .where(F.col(coluna_arquivo).isNotNull())
        .dropDuplicates()
    )

    return df_novo.join(df_processados, on=coluna_arquivo, how="left_anti")


def salvar_delta_append(df, nome_tabela: str):
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(nome_tabela)
    )
    print(f"Dados gravados em {nome_tabela}")


## Ler Bronze e selecionar apenas arquivos ainda não processados na Silver

In [0]:
df_bronze_clientes = ler_tabela_delta(TABELA_BRONZE_CLIENTES)

if "bronze_source_file" not in df_bronze_clientes.columns:
    raise Exception("A Bronze precisa conter a coluna bronze_source_file.")

if "bronze_ingested_at" not in df_bronze_clientes.columns:
    raise Exception("A Bronze precisa conter a coluna bronze_ingested_at.")

df_micro_lote = anti_duplicidade_por_arquivo(
    df_novo=df_bronze_clientes,
    tabela_destino=TABELA_SILVER,
    coluna_arquivo="bronze_source_file"
)

qtd_micro_lote = df_micro_lote.count()
print("Registros novos para processar:", qtd_micro_lote)

if qtd_micro_lote == 0:
    dbutils.notebook.exit("Nenhum arquivo novo para processar na Silver.")

display(
    df_micro_lote
    .select("bronze_source_file")
    .dropDuplicates()
    .orderBy("bronze_source_file")
)


## 3. Ler Bronze e selecionar apenas micro-lotes novos

In [0]:
df_bronze_clientes = ler_tabela_delta(TABELA_BRONZE_CLIENTES)

if "bronze_source_file" not in df_bronze_clientes.columns:
    raise Exception("A Bronze precisa conter a coluna bronze_source_file.")

if "bronze_ingested_at" not in df_bronze_clientes.columns:
    raise Exception("A Bronze precisa conter a coluna bronze_ingested_at.")

df_micro_lote = anti_duplicidade_por_arquivo(
    df_novo=df_bronze_clientes,
    tabela_destino=TABELA_SILVER,
    coluna_arquivo="bronze_source_file"
)

qtd_micro_lote = df_micro_lote.count()
print("Registros novos para processar:", qtd_micro_lote)

if qtd_micro_lote == 0:
    dbutils.notebook.exit("Nenhum arquivo novo para processar na Silver.")

display(
    df_micro_lote
    .select("bronze_source_file")
    .dropDuplicates()
    .orderBy("bronze_source_file")
)


## Padronização mínima para validação

A Silver pode criar campos auxiliares para validação, mas os dados originais são preservados. As flags indicam falhas.

In [0]:
df_base = (
    df_micro_lote
    .withColumn("id_cliente_str", F.trim(F.col("id_cliente").cast("string")))
    .withColumn("email_norm", F.lower(F.trim(F.col("email"))))
    .withColumn("nome_norm", F.trim(F.col("nome")))
    .withColumn("sobrenome_norm", F.trim(F.col("sobrenome")))
    .withColumn("senha_hash_norm", F.trim(F.col("senha_hash")))
    .withColumn("uuid_cliente_norm", F.lower(F.trim(F.col("uuid_cliente"))))
    .withColumn("dt_cadastro_ts", F.to_timestamp(F.col("dt_cadastro")))
    .withColumn("dt_ultima_atualizacao_ts", F.to_timestamp(F.col("dt_ultima_atualizacao")))
)


## 5. Preparar referências externas para regras 7 e 8

In [0]:
# Regra 7: cliente precisa ter ao menos 1 endereço associado
if tabela_existe(TABELA_BRONZE_ENDERECOS):
    df_enderecos_ref = (
        spark.table(TABELA_BRONZE_ENDERECOS)
        .select(F.col("id_cliente").cast("string").alias("id_cliente_str"))
        .where(F.col("id_cliente_str").isNotNull())
        .dropDuplicates()
        .withColumn("tem_endereco", F.lit(True))
    )
else:
    print(f"Aviso: {TABELA_BRONZE_ENDERECOS} não existe. Regra 7 marcará todos como falha.")
    df_enderecos_ref = spark.createDataFrame([], "id_cliente_str string, tem_endereco boolean")

# Regra 8: cliente precisa ter ao menos 1 pedido associado
if tabela_existe(TABELA_BRONZE_PEDIDOS):
    df_pedidos_ref = (
        spark.table(TABELA_BRONZE_PEDIDOS)
        .select(F.col("id_cliente").cast("string").alias("id_cliente_str"))
        .where(F.col("id_cliente_str").isNotNull())
        .dropDuplicates()
        .withColumn("tem_pedido", F.lit(True))
    )
else:
    print(f"Aviso: {TABELA_BRONZE_PEDIDOS} não existe. Regra 8 marcará todos como falha.")
    df_pedidos_ref = spark.createDataFrame([], "id_cliente_str string, tem_pedido boolean")


## 6. Aplicar as 10 regras como flags booleanas

In [0]:
w_id_cliente = Window.partitionBy("id_cliente_str")
w_email = Window.partitionBy("email_norm")
w_uuid = Window.partitionBy("uuid_cliente_norm")

regex_email = r"^[^@]+@[^@]+\.[^@]+$"
regex_uuid = r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$"
provedores_validos = ["gmail", "yahoo", "hotmail", "outlook", "uol", "bol", "terra"]

# Primeiro calcula contagens de duplicidade internas do micro-lote
# Regras 1, 2 e 10 usam essas contagens.
df_regras = (
    df_base
    .withColumn("qtd_id_cliente_no_lote", F.count("*").over(w_id_cliente))
    .withColumn("qtd_email_no_lote", F.count("*").over(w_email))
    .withColumn("qtd_uuid_no_lote", F.count("*").over(w_uuid))
    .withColumn("dominio_email", F.lower(F.split(F.col("email_norm"), "@").getItem(1)))
    .withColumn("provedor_email", F.split(F.col("dominio_email"), "\.").getItem(0))
)

# Para Regra 9, o percentual é calculado por arquivo de origem.
df_regra9_percentual = (
    df_regras
    .withColumn("provedor_reconhecido_linha", F.col("provedor_email").isin(provedores_validos))
    .groupBy("bronze_source_file")
    .agg(
        F.count("*").alias("total_regra9"),
        F.sum(F.when(F.col("provedor_reconhecido_linha"), 1).otherwise(0)).alias("qtd_provedor_reconhecido")
    )
    .withColumn(
        "perc_provedor_reconhecido",
        F.when(F.col("total_regra9") > 0, F.col("qtd_provedor_reconhecido") / F.col("total_regra9"))
         .otherwise(F.lit(0.0))
    )
    .select("bronze_source_file", "perc_provedor_reconhecido")
)

df_silver_clientes = (
    df_regras
    .join(df_enderecos_ref, on="id_cliente_str", how="left")
    .join(df_pedidos_ref, on="id_cliente_str", how="left")
    .join(df_regra9_percentual, on="bronze_source_file", how="left")
    .withColumn("tem_endereco", F.coalesce(F.col("tem_endereco"), F.lit(False)))
    .withColumn("tem_pedido", F.coalesce(F.col("tem_pedido"), F.lit(False)))
    
    # R1: id_cliente não pode ser nulo nem duplicado
    .withColumn(
        "r1_id_cliente_falhou",
        F.col("id_cliente_str").isNull() | (F.col("id_cliente_str") == "") | (F.col("qtd_id_cliente_no_lote") > 1)
    )
    
    # R2: email único e formato válido
    .withColumn(
        "r2_email_falhou",
        F.col("email_norm").isNull()
        | (F.col("email_norm") == "")
        | (~F.col("email_norm").rlike(regex_email))
        | (F.col("qtd_email_no_lote") > 1)
    )
    
    # R3: nome e sobrenome obrigatórios
    .withColumn(
        "r3_nome_sobrenome_falhou",
        F.col("nome_norm").isNull()
        | (F.col("nome_norm") == "")
        | F.col("sobrenome_norm").isNull()
        | (F.col("sobrenome_norm") == "")
    )
    
    # R4: senha_hash deve ter 64 caracteres
    .withColumn(
        "r4_senha_hash_falhou",
        F.col("senha_hash_norm").isNull() | (F.length(F.col("senha_hash_norm")) != 64)
    )
    
    # R5: dt_cadastro não pode ser nula nem futura
    .withColumn(
        "r5_dt_cadastro_falhou",
        F.col("dt_cadastro_ts").isNull() | (F.col("dt_cadastro_ts") > F.current_timestamp())
    )
    
    # R6: dt_ultima_atualizacao não pode ser anterior a dt_cadastro
    .withColumn(
        "r6_dt_ultima_atualizacao_falhou",
        F.col("dt_ultima_atualizacao_ts").isNotNull()
        & F.col("dt_cadastro_ts").isNotNull()
        & (F.col("dt_ultima_atualizacao_ts") < F.col("dt_cadastro_ts"))
    )
    
    # R7: todo cliente deve ter ao menos 1 endereço
    .withColumn("r7_cliente_sem_endereco_falhou", ~F.col("tem_endereco"))
    
    # R8: todo cliente ativo deve ter ao menos 1 pedido
    .withColumn("r8_cliente_sem_pedido_falhou", ~F.col("tem_pedido"))
    
    # R9: proporção de provedores reconhecidos por arquivo deve ser >= 95%
    .withColumn(
        "r9_proporcao_provedor_falhou",
        F.coalesce(F.col("perc_provedor_reconhecido"), F.lit(0.0)) < F.lit(0.95)
    )
    
    # R10: uuid_cliente único e formato UUID válido
    .withColumn(
        "r10_uuid_cliente_falhou",
        F.col("uuid_cliente_norm").isNull()
        | (F.col("uuid_cliente_norm") == "")
        | (~F.col("uuid_cliente_norm").rlike(regex_uuid))
        | (F.col("qtd_uuid_no_lote") > 1)
    )
    .withColumn(
        "silver_linha_valida",
        ~(
            F.col("r1_id_cliente_falhou")
            | F.col("r2_email_falhou")
            | F.col("r3_nome_sobrenome_falhou")
            | F.col("r4_senha_hash_falhou")
            | F.col("r5_dt_cadastro_falhou")
            | F.col("r6_dt_ultima_atualizacao_falhou")
            | F.col("r7_cliente_sem_endereco_falhou")
            | F.col("r8_cliente_sem_pedido_falhou")
            | F.col("r9_proporcao_provedor_falhou")
            | F.col("r10_uuid_cliente_falhou")
        )
    )
    .withColumn("silver_processed_at", F.current_timestamp())
)

# Remove colunas auxiliares de contagem, mas mantém flags, auditoria e lineage.
df_silver_clientes = df_silver_clientes.drop(
    "qtd_id_cliente_no_lote",
    "qtd_email_no_lote",
    "qtd_uuid_no_lote"
)

display(df_silver_clientes.limit(10))


## Resumo da validação

In [0]:
flags_regras = [
    "r1_id_cliente_falhou",
    "r2_email_falhou",
    "r3_nome_sobrenome_falhou",
    "r4_senha_hash_falhou",
    "r5_dt_cadastro_falhou",
    "r6_dt_ultima_atualizacao_falhou",
    "r7_cliente_sem_endereco_falhou",
    "r8_cliente_sem_pedido_falhou",
    "r9_proporcao_provedor_falhou",
    "r10_uuid_cliente_falhou",
]

exprs = [F.sum(F.when(F.col(c), 1).otherwise(0)).alias(c) for c in flags_regras]

display(df_silver_clientes.agg(*exprs))

display(df_silver_clientes.groupBy("silver_linha_valida").count())


## Gerar dq_monitoring_logs

Uma linha por regra e por arquivo de origem, com o schema acordado pela squad.

In [0]:
regras_config = [
    ("R1 - id_cliente não pode ser nulo nem duplicado", "r1_id_cliente_falhou", "Critica"),
    ("R2 - email único e formato válido", "r2_email_falhou", "Critica"),
    ("R3 - nome e sobrenome obrigatórios", "r3_nome_sobrenome_falhou", "Aviso"),
    ("R4 - senha_hash com exatamente 64 caracteres", "r4_senha_hash_falhou", "Critica"),
    ("R5 - dt_cadastro não pode ser nula nem futura", "r5_dt_cadastro_falhou", "Critica"),
    ("R6 - dt_ultima_atualizacao não pode ser anterior a dt_cadastro", "r6_dt_ultima_atualizacao_falhou", "Critica"),
    ("R7 - cliente deve ter ao menos 1 endereço associado", "r7_cliente_sem_endereco_falhou", "Critica"),
    ("R8 - cliente deve ter ao menos 1 pedido associado", "r8_cliente_sem_pedido_falhou", "Aviso"),
    ("R9 - proporção de provedores reconhecidos deve ser >= 95%", "r9_proporcao_provedor_falhou", "Aviso"),
    ("R10 - uuid_cliente único e formato UUID válido", "r10_uuid_cliente_falhou", "Critica"),
]


def criar_log_regra(df, nome_regra, coluna_flag, severidade):
    return (
        df
        .groupBy("bronze_source_file")
        .agg(
            F.count("*").cast("int").alias("qtd_registros_total"),
            F.sum(F.when(F.col(coluna_flag), 1).otherwise(0)).cast("int").alias("qtd_registros_falhos")
        )
        .withColumn("run_id", F.lit(RUN_ID))
        .withColumn("tabela", F.lit("silver_ecommerce_clientes"))
        .withColumn("regra", F.lit(nome_regra))
        .withColumn("status", F.when(F.col("qtd_registros_falhos") > 0, F.lit("FAIL")).otherwise(F.lit("PASS")))
        .withColumn("severidade", F.lit(severidade))
        .withColumn("timestamp_execucao", F.current_timestamp())
        .withColumnRenamed("bronze_source_file", "arquivo_origem")
        .select(
            "run_id",
            "tabela",
            "regra",
            "status",
            "severidade",
            "qtd_registros_falhos",
            "qtd_registros_total",
            "timestamp_execucao",
            "arquivo_origem"
        )
    )

logs = [criar_log_regra(df_silver_clientes, regra, flag, severidade) for regra, flag, severidade in regras_config]
df_dq_monitoring_logs = reduce(lambda a, b: a.unionByName(b), logs)

display(df_dq_monitoring_logs.orderBy("arquivo_origem", "regra"))


## Gravar Silver e dq_monitoring_logs em Delta

In [0]:

from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)

DQ_LOGS_TABLE = TABELA_DQ_LOGS

schema_dq_logs = StructType([
    StructField("run_id", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("regra", StringType(), False),
    StructField("status", StringType(), False),
    StructField("severidade", StringType(), False),
    StructField("qtd_registros_falhos", IntegerType(), False),
    StructField("qtd_registros_total", IntegerType(), False),
    StructField("timestamp_execucao", TimestampType(), False),
    StructField("arquivo_origem", StringType(), False),
])

if not spark.catalog.tableExists(DQ_LOGS_TABLE):
    df_empty_logs = spark.createDataFrame([], schema_dq_logs)

    (
        df_empty_logs.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(DQ_LOGS_TABLE)
    )

    print(f"Tabela Delta criada: {DQ_LOGS_TABLE}")
else:
    print(f"Tabela Delta já existe: {DQ_LOGS_TABLE}")


In [0]:
salvar_delta_append(df_silver_clientes, TABELA_SILVER)



## Validação final

In [0]:
print("=" * 80)
print("SILVER CONCLUÍDA")
print("RUN_ID:", RUN_ID)
print("Registros processados na Silver:", df_silver_clientes.count())
print("Logs DQ gerados:", df_dq_monitoring_logs.count())
print("Tabela Silver:", TABELA_SILVER)
print("Tabela Logs:", TABELA_DQ_LOGS)
print("=" * 80)

print("Arquivos processados neste run:")
display(df_silver_clientes.select("bronze_source_file").dropDuplicates().orderBy("bronze_source_file"))

print("Resumo DQ do run:")
display(
    df_dq_monitoring_logs
    .groupBy("status", "severidade")
    .agg(F.sum("qtd_registros_falhos").alias("falhas"), F.count("*").alias("logs"))
)
